# LeetCode 2843 — Count Symmetric Integers bằng Python

Link đề: https://leetcode.com/problems/count-symmetric-integers/ · Độ khó: **Easy** · Chủ đề: Math, Enumeration, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def countSymmetricIntegers(self, low: int, high: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một số nguyên có `2n` chữ số là **đối xứng (symmetric)** nếu **tổng `n` chữ số đầu bằng tổng `n` chữ số cuối**; số có **số lẻ** chữ số thì **không bao giờ** đối xứng. Đếm các số đối xứng trong đoạn `[low, high]`.
2. Biết **tự tính tay**: chia số thành hai nửa, cộng các chữ số mỗi nửa, so sánh.
3. Học các công cụ Python: `str(x)`, `len`, cắt lát `s[:h]` / `s[h:]`, `map(int, ...)` và `sum`, bóc chữ số bằng `//` và `%`, mảng cộng dồn (prefix sum), `bisect`.
4. Đi từ **Brute Force dùng chuỗi cho mỗi số** → **bóc chữ số bằng toán (không chuỗi)** → **tiền xử lý một lần + prefix sum (trả lời nhiều truy vấn)** → **sinh trực tiếp các số đối xứng rồi đếm bằng `bisect`**.
5. Hiểu vì sao với `high <= 10^4` chỉ cần xét số có **2 hoặc 4 chữ số**, và tránh các lỗi kinh điển: coi số lẻ chữ số là đối xứng, quên rằng `10000` có 5 chữ số, lỗi lệch biên `[low, high]` (bao gồm cả hai đầu).

## 2. Đề bài & ví dụ

Bạn được cho hai số nguyên dương `low` và `high`. Một số nguyên `x` gồm `2 × n` chữ số được gọi là **đối xứng** nếu **tổng `n` chữ số đầu** của `x` **bằng** **tổng `n` chữ số cuối** của `x`. Các số có **số lượng chữ số lẻ** không bao giờ là đối xứng.

Trả về **số lượng** số nguyên đối xứng trong đoạn **`[low, high]`** (tính cả `low` và `high`).

| Ví dụ | `low`, `high` | Output | Giải thích |
|---|---|---|---|
| 1 | `1`, `100` | `9` | Có 9 số đối xứng trong đoạn `[1, 100]`: `11, 22, 33, 44, 55, 66, 77, 88, 99`. |
| 2 | `1200`, `1230` | `4` | Có 4 số đối xứng trong `[1200, 1230]`: `1203, 1212, 1221, 1230`. |

Ví dụ chi tiết: `1203` có hai nửa là `12` và `03`: `1 + 2 = 3` và `0 + 3 = 3` ⇒ đối xứng. `1230` có hai nửa `12` và `30`: `1 + 2 = 3`, `3 + 0 = 3` ⇒ đối xứng.

Constraints:

- `1 <= low <= high <= 10^4`

In [1]:
# Kiểm tay vài số: hai nửa và tổng chữ số mỗi nửa
for x in (11, 12, 1203, 1230, 1231, 123):
    s = str(x)
    if len(s) % 2 == 1:
        print(x, "-> số chữ số lẻ, không đối xứng")
    else:
        h = len(s) // 2
        left, right = s[:h], s[h:]
        print(x, "->", left, right, sum(map(int, left)), sum(map(int, right)))

11 -> 1 1 1 1
12 -> 1 2 1 2
1203 -> 12 03 3 3
1230 -> 12 30 3 3
1231 -> 12 31 3 4
123 -> số chữ số lẻ, không đối xứng


## 3. Kiến thức nền

### 3.1. Số chữ số và tính chẵn lẻ

`len(str(x))` là số chữ số của `x`. `n % 2 == 0` kiểm số chẵn. Số có số chữ số **lẻ** (1, 3, 5, ...) bị loại ngay theo đề.

In [2]:
print(len(str(7)), len(str(99)), len(str(100)), len(str(10000)))   # 1 2 3 5
print(len(str(1203)) % 2 == 0)                                      # True

1 2 3 5
True


### 3.2. Cắt lát chuỗi: `s[:h]` và `s[h:]`

`s[:h]` lấy `h` ký tự **đầu**; `s[h:]` lấy phần còn lại từ vị trí `h` trở đi. Với chuỗi độ dài `2h`, đây chính là **hai nửa** bằng nhau.

In [3]:
s = "1203"
h = len(s) // 2
print(s[:h], s[h:])      # 12 03
print("123456"[:3], "123456"[3:])   # 123 456

12 03
123 456


### 3.3. `sum`, `map(int, ...)` — tổng các chữ số

`map(int, "12")` biến từng ký tự thành số nguyên (lười); `sum(...)` cộng chúng. `sum(map(int, "1203"))` = `1 + 2 + 0 + 3`.

In [4]:
print(sum(map(int, "12")))        # 3
print(sum(map(int, "03")))        # 3
print(sum(int(c) for c in "1203"))  # 6  (tương đương)

3
3
6


### 3.4. Bóc chữ số bằng `//` và `%` (không dùng chuỗi)

Với số có **4 chữ số** `x`: `a = x // 1000` (chữ số nghìn), `b = x // 100 % 10`, `c = x // 10 % 10`, `d = x % 10`. Với số **2 chữ số**: `a = x // 10`, `b = x % 10`.

In [5]:
x = 1203
a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
print(a, b, c, d)         # 1 2 0 3
print(a + b == c + d)     # True

1 2 0 3
True


### 3.5. Mảng cộng dồn (prefix sum)

Mảng `prefix` với `prefix[i]` = tổng của `f(1) + ... + f(i)` cho phép tính tổng trên một đoạn `[l, r]` trong `O(1)`: `prefix[r] − prefix[l − 1]`. Ở đây `f(x) = 1` nếu `x` đối xứng, ngược lại `0`.

In [6]:
f = [0, 1, 0, 1, 1, 0]            # chỉ số 0..5 (f[0] không dùng)
prefix = [0] * len(f)
for i in range(1, len(f)):
    prefix[i] = prefix[i - 1] + f[i]
print(prefix)                      # [0, 1, 1, 2, 3, 3]
l, r = 2, 4
print(prefix[r] - prefix[l - 1])   # f[2] + f[3] + f[4] = 0 + 1 + 1 = 2

[0, 1, 1, 2, 3, 3]
2


### 3.6. `bisect_left` / `bisect_right` — đếm phần tử trong đoạn của list đã sắp xếp

Với list **đã sắp xếp** `a`: số phần tử `x` thoả `l <= x <= r` là `bisect_right(a, r) − bisect_left(a, l)`. Độ phức tạp `O(log n)`.

In [7]:
from bisect import bisect_left, bisect_right

a = [11, 22, 33, 44]
print(bisect_right(a, 33) - bisect_left(a, 22))   # 2  (22 và 33)
print(bisect_right(a, 10) - bisect_left(a, 1))    # 0

2
0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 2: đoạn `[1200, 1230]` (31 số). Với mỗi số, chữ số nghìn–trăm là `1, 2` ⇒ tổng nửa đầu `1 + 2 = 3`. Số `12cd` đối xứng ⇔ `c + d = 3`:

| `c d` | số | `c + d` | đối xứng? |
|---|---|---|---|
| 0 3 | 1203 | 3 | có |
| 1 2 | 1212 | 3 | có |
| 2 1 | 1221 | 3 | có |
| 3 0 | 1230 | 3 | có |
| 0 0 | 1200 | 0 | không |
| 2 5 | 1225 | 7 | không |

Trong đoạn `[1200, 1230]`, số `12cd` có `cd` chạy `00..30`. Những `cd` có `c + d = 3` là `03, 12, 21, 30` ⇒ **4** số ✓.

**Nhận xét:**

- Số có **1 chữ số** (1–9) và **3 chữ số** (100–999) và **5 chữ số** (`10000`) có số chữ số **lẻ** ⇒ không bao giờ đối xứng.
- Số **2 chữ số** `ab` đối xứng ⇔ `a == b`, tức `11, 22, ..., 99`.
- Số **4 chữ số** `abcd` đối xứng ⇔ `a + b == c + d`.

Vì `high <= 10^4`, mọi số đối xứng có 2 hoặc 4 chữ số.

In [8]:
# Đếm các số 2 chữ số đối xứng và vài số 4 chữ số 12cd
print([x for x in range(10, 100) if x // 10 == x % 10])
print([x for x in range(1200, 1231) if (x // 1000) + (x // 100 % 10) == (x // 10 % 10) + (x % 10)])

[11, 22, 33, 44, 55, 66, 77, 88, 99]
[1203, 1212, 1221, 1230]


## 5. Cách 1 — Brute Force / baseline: duyệt từng số, dùng chuỗi

**Ý tưởng:** với mỗi `x` từ `low` tới `high`:

1. `s = str(x)`; nếu `len(s)` lẻ thì bỏ qua.
2. `h = len(s) // 2`; so sánh `sum(map(int, s[:h]))` với `sum(map(int, s[h:]))`.
3. Bằng nhau thì tăng `count`.

Bản dịch thẳng của đề, rất phù hợp làm **oracle**.

In [9]:
from typing import List

def is_symmetric_str(x: int) -> bool:
    s = str(x)
    if len(s) % 2 == 1:
        return False
    h = len(s) // 2
    return sum(map(int, s[:h])) == sum(map(int, s[h:]))

def count_bruteforce(low: int, high: int) -> int:
    count = 0
    for x in range(low, high + 1):
        if is_symmetric_str(x):
            count += 1
    return count

In [10]:
print(count_bruteforce(1, 100))        # 9
print(count_bruteforce(1200, 1230))    # 4
print(is_symmetric_str(10000), is_symmetric_str(1203), is_symmetric_str(123))   # False True False

9
4
False True False


**Độ phức tạp:** gọi `m = high − low + 1` là số số cần duyệt và `d` là số chữ số (≤ 5). Mỗi số xử lý `O(d)` ⇒ Time `O(m·d)`. Space `O(d)` cho chuỗi tạm.

**Nút thắt:** tạo chuỗi và `map(int, ...)` cho từng số khá tốn kém. Hơn nữa, nếu có **nhiều truy vấn** `(low, high)` thì ta lặp lại công việc cho mỗi truy vấn.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ cần xét số 2 và 4 chữ số.** Số có số chữ số lẻ bị loại ngay; với `x <= 10^4` chỉ còn 2 và 4 chữ số. Có thể dùng điều kiện toán học trực tiếp: số 2 chữ số `x` đối xứng ⇔ `x % 11 == 0` (với `10 <= x <= 99`); số 4 chữ số ⇔ `a + b == c + d`.

**Đột phá 2 — bỏ chuỗi, bóc chữ số bằng `//` và `%`.** Nhanh hơn, không cấp phát.

**Đột phá 3 — tiền xử lý một lần.** Miền giá trị chỉ `1..10^4`. Tạo mảng `prefix[i]` = số số đối xứng trong `[1, i]` một lần (`O(10^4)`), rồi mỗi truy vấn đếm trong `O(1)`: `prefix[high] − prefix[low − 1]`.

**Đột phá 4 — sinh trực tiếp.** Số đối xứng 4 chữ số `abcd` ứng với bộ `(a, b, c, d)` thoả `a ≥ 1` và `a + b = c + d`. Ta có thể **sinh** tất cả số đối xứng (chỉ khoảng 600 số) rồi đếm những số nằm trong `[low, high]` bằng `bisect` mà không phải duyệt cả đoạn.

In [11]:
print("số 2 chữ số: x % 11 == 0:", [x for x in range(10, 100) if x % 11 == 0])
cnt4 = 0
for a in range(1, 10):
    for b in range(10):
        for c in range(10):
            for d in range(10):
                if a + b == c + d:
                    cnt4 += 1
print("số 4 chữ số đối xứng:", cnt4)

số 2 chữ số: x % 11 == 0: [11, 22, 33, 44, 55, 66, 77, 88, 99]
số 4 chữ số đối xứng: 615


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Bóc chữ số bằng toán, không dùng chuỗi

**Từng đoạn:**

- `10 <= x <= 99`: `x // 10 == x % 10`.
- `1000 <= x <= 9999`: `a = x // 1000`, `b = x // 100 % 10`, `c = x // 10 % 10`, `d = x % 10`; kiểm `a + b == c + d`.
- Các trường hợp khác (1, 3, 5 chữ số): không đối xứng.

In [12]:
def is_symmetric_math(x: int) -> bool:
    if 10 <= x <= 99:
        return x // 10 == x % 10
    if 1000 <= x <= 9999:
        a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
        return a + b == c + d
    return False

def count_math(low: int, high: int) -> int:
    return sum(is_symmetric_math(x) for x in range(low, high + 1))

In [13]:
print(count_math(1, 100))        # 9
print(count_math(1200, 1230))    # 4

9
4


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def countSymmetricIntegers(self, low: int, high: int) -> int:
        count = 0
        for x in range(low, high + 1):
            if 10 <= x <= 99:
                if x // 10 == x % 10:
                    count += 1
            elif 1000 <= x <= 9999:
                a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
                if a + b == c + d:
                    count += 1
        return count
```

**Độ phức tạp:** Time `O(m)` (`m = high − low + 1`), mỗi số `O(1)`; Space `O(1)`.

### Cách 3 — Prefix sum: tiền xử lý một lần cho nhiều truy vấn

**Từng đoạn:**

- `LIMIT = 10_000`: giá trị tối đa.
- `prefix[i]` = số số đối xứng trong `[1, i]`: `prefix[i] = prefix[i − 1] + (1 nếu i đối xứng)`.
- Truy vấn `(low, high)`: `prefix[high] − prefix[low − 1]`.

In [14]:
LIMIT = 10_000
prefix = [0] * (LIMIT + 1)
for i in range(1, LIMIT + 1):
    prefix[i] = prefix[i - 1] + (1 if is_symmetric_math(i) else 0)

def count_prefix(low: int, high: int) -> int:
    return prefix[high] - prefix[low - 1]

In [15]:
print(count_prefix(1, 100))        # 9
print(count_prefix(1200, 1230))    # 4
print(prefix[LIMIT])               # tổng số số đối xứng trong [1, 10000]

9
4
624


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def countSymmetricIntegers(self, low: int, high: int) -> int:
        def sym(x):
            if 10 <= x <= 99:
                return x // 10 == x % 10
            if 1000 <= x <= 9999:
                a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
                return a + b == c + d
            return False

        prefix = [0] * (high + 1)
        for i in range(1, high + 1):
            prefix[i] = prefix[i - 1] + (1 if sym(i) else 0)
        return prefix[high] - prefix[low - 1]
```

**Độ phức tạp:** tiền xử lý `O(10^4)`, mỗi truy vấn `O(1)`. Với **một** truy vấn như đề, tổng vẫn `O(high)`; lợi thế rõ khi có nhiều truy vấn trên cùng miền.

### Cách 4 — Sinh trực tiếp các số đối xứng rồi đếm bằng `bisect`

**Từng đoạn:**

- Sinh số 2 chữ số: `11 * a` với `a = 1..9`.
- Sinh số 4 chữ số: duyệt `(a, b, c, d)` với `a ≥ 1`, `a + b == c + d`, giá trị `1000a + 100b + 10c + d`. Thu vào list, rồi **sắp xếp** (để `bisect` dùng được).
- Đếm trong đoạn: `bisect_right(lst, high) − bisect_left(lst, low)`.

In [16]:
def generate_symmetric() -> List[int]:
    res = [11 * a for a in range(1, 10)]                  # 11, 22, ..., 99
    for a in range(1, 10):
        for b in range(10):
            for c in range(10):
                d = a + b - c                                 # ép a + b == c + d
                if 0 <= d <= 9:
                    res.append(1000 * a + 100 * b + 10 * c + d)
    res.sort()
    return res

SYM = generate_symmetric()

def count_generate(low: int, high: int) -> int:
    return bisect_right(SYM, high) - bisect_left(SYM, low)

In [17]:
print(len(SYM), SYM[:12])
print(count_generate(1, 100))        # 9
print(count_generate(1200, 1230))    # 4

624 [11, 22, 33, 44, 55, 66, 77, 88, 99, 1001, 1010, 1102]
9
4


**Giải thích điều kiện `d = a + b − c`:** thay vì duyệt cả `d` rồi kiểm `a + b == c + d`, ta suy ra `d` từ `a, b, c`; chỉ giữ nếu `d` là một chữ số (`0..9`). Giảm 10 lần số vòng lặp.

**Độ phức tạp:** sinh một lần `O(9·10·10)` = hằng số; mỗi truy vấn `O(log |SYM|)` ≈ `O(1)`. Space `O(|SYM|)` ≈ 624 phần tử.

## 8. TRACE từng bước

Trace Cách 2 trên đoạn `[1200, 1230]`: ta chỉ liệt kê 12 số đầu để bảng gọn.

| `x` | `a, b, c, d` | `a + b` | `c + d` | đối xứng? | `count` sau |
|---|---|---|---|---|---|
| 1200 | 1, 2, 0, 0 | 3 | 0 | không | 0 |
| 1201 | 1, 2, 0, 1 | 3 | 1 | không | 0 |
| 1202 | 1, 2, 0, 2 | 3 | 2 | không | 0 |
| 1203 | 1, 2, 0, 3 | 3 | 3 | **có** | 1 |
| 1204 | 1, 2, 0, 4 | 3 | 4 | không | 1 |
| ... | | | | | |
| 1212 | 1, 2, 1, 2 | 3 | 3 | **có** | 2 |
| 1221 | 1, 2, 2, 1 | 3 | 3 | **có** | 3 |
| 1230 | 1, 2, 3, 0 | 3 | 3 | **có** | 4 |

In [18]:
def trace_count(low, high, limit_print=40):
    count = 0
    in_ra = 0
    for x in range(low, high + 1):
        if 10 <= x <= 99:
            a, b = x // 10, x % 10
            ok = a == b
            chi_tiet = f"2 chữ số: {a} vs {b}"
        elif 1000 <= x <= 9999:
            a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
            ok = a + b == c + d
            chi_tiet = f"4 chữ số: {a}+{b}={a + b} vs {c}+{d}={c + d}"
        else:
            ok = False
            chi_tiet = "số chữ số lẻ -> bỏ"
        count += ok
        if ok or in_ra < limit_print:
            print(f"x={x:5} {chi_tiet:28} -> {'ĐỐI XỨNG' if ok else 'không'} | count={count}")
            in_ra += 1
    print("kết quả:", count)
    return count

trace_count(1200, 1230, limit_print=5)

x= 1200 4 chữ số: 1+2=3 vs 0+0=0     -> không | count=0
x= 1201 4 chữ số: 1+2=3 vs 0+1=1     -> không | count=0
x= 1202 4 chữ số: 1+2=3 vs 0+2=2     -> không | count=0
x= 1203 4 chữ số: 1+2=3 vs 0+3=3     -> ĐỐI XỨNG | count=1
x= 1204 4 chữ số: 1+2=3 vs 0+4=4     -> không | count=1
x= 1212 4 chữ số: 1+2=3 vs 1+2=3     -> ĐỐI XỨNG | count=2
x= 1221 4 chữ số: 1+2=3 vs 2+1=3     -> ĐỐI XỨNG | count=3
x= 1230 4 chữ số: 1+2=3 vs 3+0=3     -> ĐỐI XỨNG | count=4
kết quả: 4


4

In [19]:
trace_count(8, 25, limit_print=20)

x=    8 số chữ số lẻ -> bỏ           -> không | count=0
x=    9 số chữ số lẻ -> bỏ           -> không | count=0
x=   10 2 chữ số: 1 vs 0             -> không | count=0
x=   11 2 chữ số: 1 vs 1             -> ĐỐI XỨNG | count=1
x=   12 2 chữ số: 1 vs 2             -> không | count=1
x=   13 2 chữ số: 1 vs 3             -> không | count=1
x=   14 2 chữ số: 1 vs 4             -> không | count=1
x=   15 2 chữ số: 1 vs 5             -> không | count=1
x=   16 2 chữ số: 1 vs 6             -> không | count=1
x=   17 2 chữ số: 1 vs 7             -> không | count=1
x=   18 2 chữ số: 1 vs 8             -> không | count=1
x=   19 2 chữ số: 1 vs 9             -> không | count=1
x=   20 2 chữ số: 2 vs 0             -> không | count=1
x=   21 2 chữ số: 2 vs 1             -> không | count=1
x=   22 2 chữ số: 2 vs 2             -> ĐỐI XỨNG | count=2
x=   23 2 chữ số: 2 vs 3             -> không | count=2
x=   24 2 chữ số: 2 vs 4             -> không | count=2
x=   25 2 chữ số: 2 vs 5             -> kh

2

## 9. Code nộp LeetCode

Dùng Cách 2 (bóc chữ số bằng toán): gọn, `O(high − low)`, đủ cho ràng buộc `10^4`.

In [20]:
class Solution:
    def countSymmetricIntegers(self, low: int, high: int) -> int:
        count = 0
        for x in range(low, high + 1):
            if 10 <= x <= 99:                       # số 2 chữ số: ab với a == b
                if x // 10 == x % 10:
                    count += 1
            elif 1000 <= x <= 9999:                 # số 4 chữ số: a + b == c + d
                a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
                if a + b == c + d:
                    count += 1
        return count

In [21]:
print(Solution().countSymmetricIntegers(1, 100))        # 9
print(Solution().countSymmetricIntegers(1200, 1230))    # 4

9
4


## 10. Vì sao thuật toán đúng

**Bước 1 — chỉ số có số chữ số chẵn mới có thể đối xứng.** Theo đề, số có số chữ số lẻ không bao giờ đối xứng. Với `1 <= x <= 10^4`, số chữ số của `x` là `1, 2, 3, 4` hoặc `5` (chỉ `x = 10000`). Các độ dài chẵn là `2` và `4`.

**Bước 2 — số 2 chữ số.** `x = 10a + b` với `a ∈ 1..9`, `b ∈ 0..9`. `n = 1`: nửa đầu chỉ có `a`, nửa sau chỉ có `b`. Đối xứng ⇔ `a = b`. Mã `x // 10 == x % 10` đúng ý.

**Bước 3 — số 4 chữ số.** `x = 1000a + 100b + 10c + d` với `a ∈ 1..9`. `n = 2`: nửa đầu có `a, b`, nửa sau có `c, d` (kể cả `c = 0`, ví dụ `1203` có nửa sau `03`). Đối xứng ⇔ `a + b = c + d`. Điều kiện `1000 <= x <= 9999` bảo đảm đúng 4 chữ số.

**Bước 4 — bao gồm hai đầu.** Vòng `range(low, high + 1)` duyệt cả `low` và `high` (đề yêu cầu đoạn đóng `[low, high]`).

**Bước 5 — đối chiếu.** Cả 4 cách đếm cùng một tập số đối xứng; ta kiểm bằng vét cạn toàn bộ `(low, high)` ở mục 12.

## 11. Phân tích Time / Space Complexity

Gọi `m = high − low + 1`, `M = 10^4` (miền tối đa).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Chuỗi cho từng số | `O(m·d)` | `O(d)` | `d` ≤ 5 chữ số; oracle |
| 2. Bóc chữ số bằng toán | `O(m)` | `O(1)` | **code nộp** |
| 3. Prefix sum | tiền xử lý `O(M)`, mỗi truy vấn `O(1)` | `O(M)` | nhiều truy vấn |
| 4. Sinh + `bisect` | tiền xử lý hằng số (~900 vòng lặp), mỗi truy vấn `O(log 624)` | `O(624)` | nhanh nhất cho nhiều truy vấn |

Với một truy vấn và `m <= 10^4`, mọi cách chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: `low = high` (một số); số 1 chữ số; số 3 chữ số (toàn bộ không đối xứng); `10000` (5 chữ số); biên `99` / `100`; biên `1000`; số có chữ số `0` trong nửa sau như `1203`.

In [22]:
impls = {
    "bruteforce": count_bruteforce,
    "math": count_math,
    "prefix": count_prefix,
    "generate": count_generate,
    "Solution": Solution().countSymmetricIntegers,
}
cases = [
    ("ví dụ 1", 1, 100, 9),
    ("ví dụ 2", 1200, 1230, 4),
    ("một số đối xứng", 1203, 1203, 1),
    ("một số không đối xứng", 1204, 1204, 0),
    ("một số 1 chữ số", 5, 5, 0),
    ("toàn số 3 chữ số", 100, 999, 0),
    ("10000 không đối xứng", 10000, 10000, 0),
    ("biên 99 và 100", 99, 100, 1),
    ("biên 1000", 1000, 1000, 0),
    ("chỉ số 2 chữ số", 10, 99, 9),
    ("1 đến 10000", 1, 10000, None),   # tính bằng brute force bên dưới
]
for name, low, high, exp in cases:
    if exp is None:
        exp = count_bruteforce(low, high)
    for k, f in impls.items():
        got = f(low, high)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 9
PASS ví dụ 2 -> 4
PASS một số đối xứng -> 1
PASS một số không đối xứng -> 0
PASS một số 1 chữ số -> 0
PASS toàn số 3 chữ số -> 0
PASS 10000 không đối xứng -> 0
PASS biên 99 và 100 -> 1
PASS biên 1000 -> 0
PASS chỉ số 2 chữ số -> 9
PASS 1 đến 10000 -> 624


Kiểm tay: `1000` có nửa đầu `1 + 0 = 1`, nửa sau `0 + 0 = 0` ⇒ không đối xứng ✓. `[99, 100]` chỉ có `99` ✓. Ca `[1, 10000]` có đáp án là tổng số số đối xứng (ta lấy từ brute force và so với các cách còn lại).

### 12.2. Đối chiếu nhiều đoạn ngẫu nhiên có seed

Input hợp lệ theo constraints: `1 <= low <= high <= 10^4`. Ta kiểm 1500 đoạn ngẫu nhiên với các độ dài khác nhau.

In [23]:
import random
random.seed(2843)
for _ in range(1500):
    low = random.randint(1, 10000)
    high = random.randint(low, min(10000, low + random.choice([0, 5, 50, 500, 10000])))
    exp = count_bruteforce(low, high)
    for k, f in impls.items():
        assert f(low, high) == exp, (low, high, k)
print("1500 đoạn ngẫu nhiên: mọi cách khớp brute force")

1500 đoạn ngẫu nhiên: mọi cách khớp brute force


### 12.3. Kiểm tính chất

- **Cộng được:** `count(low, high) = count(low, mid) + count(mid + 1, high)` với `low <= mid < high`.
- **Không giảm khi mở rộng đoạn:** mở `high` lớn hơn không làm đáp án giảm.
- Tổng số số đối xứng trong `[1, 10000]` bằng `9 + (số bộ (a,b,c,d) thoả a ≥ 1, a + b = c + d)`.

In [24]:
random.seed(28430)
for _ in range(300):
    low = random.randint(1, 9999)
    high = random.randint(low + 1, 10000)
    mid = random.randint(low, high - 1)
    S = Solution()
    assert S.countSymmetricIntegers(low, high) == S.countSymmetricIntegers(low, mid) + S.countSymmetricIntegers(mid + 1, high)
    assert S.countSymmetricIntegers(low, high) >= S.countSymmetricIntegers(low, mid)

tong_4 = sum(1 for a in range(1, 10) for b in range(10) for c in range(10) for d in range(10) if a + b == c + d)
assert Solution().countSymmetricIntegers(1, 10000) == 9 + tong_4
print("tổng số số đối xứng trong [1, 10000] =", 9 + tong_4)

tổng số số đối xứng trong [1, 10000] = 624


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi số có số chữ số lẻ là đối xứng bằng cách chia lệch

In [25]:
def sai_so_le(x):
    s = str(x)
    h = len(s) // 2                       # với 3 chữ số h = 1: nửa đầu 1, nửa sau 2 chữ số
    return sum(map(int, s[:h])) == sum(map(int, s[h:]))

print("sai :", sai_so_le(110), sai_so_le(220))                       # True True (sai)
print("đúng:", is_symmetric_str(110), is_symmetric_str(220))           # False False

sai : True True
đúng: False False


**Vì sao sai:** đề quy định số có **số chữ số lẻ** **không bao giờ** đối xứng; phải kiểm `len(s) % 2 == 0` trước. Chia `len // 2` với số lẻ chữ số cho hai nửa không cùng độ dài nhưng tổng vẫn có thể vô tình bằng nhau (`110`: `1` và `1 + 0`).

### Lỗi 2: Áp công thức 4 chữ số cho mọi số

In [26]:
def sai_cong_thuc_4(x):
    a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
    return a + b == c + d                # chỉ đúng cho số đúng 4 chữ số

print("sai :", sai_cong_thuc_4(11), sai_cong_thuc_4(1))              # 11: a=b=0, c=d... -> False, 1: 0==1 -> False
print("đúng:", Solution().countSymmetricIntegers(11, 11), "(số 11 là đối xứng)")
print("sai :", sai_cong_thuc_4(100), "| đúng:", is_symmetric_str(100))   # True (sai) vs False

sai : False False
đúng: 1 (số 11 là đối xứng)
sai : False | đúng: False


**Vì sao sai:** công thức `a + b == c + d` giả định **đúng 4 chữ số**. Với `11` nó cho `0 + 0 == 1 + 1` là `False` (bỏ sót số đối xứng 2 chữ số), còn với `100` cho `0 + 0 == 0 + 0` là `True` (nhận nhầm số 3 chữ số). Phải **chia theo số chữ số** như trong code nộp: `10 <= x <= 99` và `1000 <= x <= 9999`.

### Lỗi 3: Lệch biên — dùng `range(low, high)` thay vì `range(low, high + 1)`

In [27]:
def sai_lech_bien(low, high):
    return sum(is_symmetric_str(x) for x in range(low, high))     # thiếu high

print("sai :", sai_lech_bien(1, 99))               # 8 (thiếu 99)
print("đúng:", Solution().countSymmetricIntegers(1, 99))   # 9

sai : 8
đúng: 9


**Vì sao sai:** đoạn `[low, high]` là **đóng**; `range(low, high)` bỏ `high`. Cần `range(low, high + 1)`.

### Lỗi 4: So sánh chữ số với tổng của nửa còn lại bằng chuỗi

In [28]:
def sai_so_sanh_chuoi(x):
    s = str(x)
    if len(s) % 2:
        return False
    h = len(s) // 2
    return s[:h] == s[h:]               # sai: so hai nửa, không so TỔNG chữ số

print("sai :", sai_so_sanh_chuoi(1203), sai_so_sanh_chuoi(1212))          # False True
print("đúng:", is_symmetric_str(1203), is_symmetric_str(1212))            # True True

sai : False True
đúng: True True


**Vì sao sai:** "đối xứng" ở đây là **tổng chữ số hai nửa bằng nhau**, không phải hai nửa **giống nhau** (đó là bài khác). `1203` đối xứng dù hai nửa `12` và `03` khác nhau.

### Lỗi 5 (nhỏ): Dùng `x // 100 // 10` thay cho `x // 100 % 10`

`x // 100` của `1203` là `12`, chứ không phải chữ số trăm `2`. Phải lấy phần dư `% 10` để chỉ giữ một chữ số.

In [29]:
x = 1203
print(x // 100, x // 100 % 10)    # 12 2

12 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Khi miền giá trị **nhỏ** (`<= 10^4`) và ràng buộc "kiểu số học", **vét cạn** từng số là cách an toàn; sau đó nghĩ tới tiền xử lý.
- Đề liên quan "số chữ số chẵn / lẻ": liệt kê các độ dài khả dĩ (ở đây chỉ 2 và 4) để viết điều kiện gọn.
- **Prefix sum** biến "đếm trên đoạn" thành hiệu hai giá trị tiền xử lý: `prefix[high] − prefix[low − 1]`.
- Khi tập các số thoả mãn **ít** (khoảng 624 số), có thể **sinh trực tiếp** rồi dùng `bisect` để đếm trong đoạn.
- Luôn kiểm các **biên đóng** của đoạn: `low`, `high` đều được tính.

## 15. Biến thể của bài

1. **Tăng giới hạn** `high <= 10^9`: dùng digit DP hoặc đếm theo độ dài; chuỗi `str` không đủ nhanh nếu duyệt từng số.
2. **Đếm số có tổng chữ số hai nửa chênh nhau đúng `k`.**
3. **Palindrome** (đối xứng gương): so hai nửa **đảo** nhau; tính chất khác hoàn toàn.
4. **Đếm số đẹp khác** (như LeetCode 2828 / 1399 "đếm nhóm theo tổng chữ số").
5. **Nhiều truy vấn** `Q` đoạn: prefix sum cho `O(M + Q)`.
6. **Chia ba phần:** tổng ba đoạn chữ số bằng nhau (số có `3n` chữ số).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi từng số | `str`, cắt đôi, `sum(map(int, ...))` | `O(m·d)` | `O(d)` | oracle, dễ hiểu |
| 2. Bóc chữ số bằng toán | `a + b == c + d` hoặc `x // 10 == x % 10` | `O(m)` | `O(1)` | **code nộp** |
| 3. Prefix sum | `prefix[high] − prefix[low − 1]` | `O(M)` + `O(1)`/truy vấn | `O(M)` | nhiều truy vấn |
| 4. Sinh + `bisect` | sinh mọi số đối xứng rồi đếm | hằng số + `O(log)` | `O(624)` | nhiều truy vấn, miền nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len(str(2025))`, `"2025"[:2]`, `"2025"[2:]`, `2025 // 100 % 10`.

**Bài 2 (tính tay).** Liệt kê các số đối xứng trong `[1000, 1100]`. Gợi ý: các số `10cd` có nửa đầu `1 + 0 = 1`, nên cần `c + d = 1`; riêng `1100` có nửa đầu `1 + 1 = 2`.

**Bài 3 (điền TODO).** Hoàn thành `is_symmetric_todo(x)` bóc chữ số bằng toán.

In [30]:
def is_symmetric_todo(x):
    # TODO 1: nếu 10 <= x <= 99 thì so sánh x // 10 với x % 10
    # TODO 2: nếu 1000 <= x <= 9999 thì tách a, b, c, d và so a + b với c + d
    # TODO 3: các trường hợp khác trả False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_prefix2(queries)` nhận danh sách truy vấn `[(low, high), ...]` và trả về list đáp án, dùng mảng prefix **không nhìn** code mẫu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_symmetric_general(low, high)` cho **mọi độ dài chẵn** (dùng chuỗi, không giới hạn `10^4`). (b) Viết `count_palindromes(low, high)` đếm số **đối xứng gương** (palindrome) trong đoạn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `len("2025") = 4`; `"2025"[:2] = "20"`, `"2025"[2:] = "25"`; `2025 // 100 = 20`, `20 % 10 = 0`. Bài 2: nửa đầu của `10cd` là `1 + 0 = 1`, nên `c + d = 1`: `1001`, `1010`.
- **Mức 2 (vừa):** Bài 3: ba dòng đã ghi trong comment TODO. Bài 4: xây `prefix` đến `max(high)` rồi trả `prefix[h] - prefix[l - 1]` cho mỗi truy vấn.
- **Mức 3 (sâu):** Bài 5(a): dùng cách 1 nhưng bỏ điều kiện `x <= 10^4`; kiểm `len(s) % 2 == 0`. Bài 5(b): `str(x) == str(x)[::-1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert len(str(2025)) == 4
assert "2025"[:2] == "20" and "2025"[2:] == "25"
assert 2025 // 100 % 10 == 0

# Bài 2: [1000, 1100]
# nhóm 10cd (1000..1099): nửa đầu 1+0 = 1, cần c+d = 1 -> cd = 01 hoặc 10 -> 1001, 1010
# 1100: nửa đầu 1+1 = 2, nửa sau 0+0 = 0 -> không đối xứng
ds = [x for x in range(1000, 1101) if is_symmetric_str(x)]
assert ds == [1001, 1010]

# Bài 3
def is_symmetric_todo_ans(x):
    if 10 <= x <= 99:
        return x // 10 == x % 10
    if 1000 <= x <= 9999:
        a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
        return a + b == c + d
    return False

assert all(is_symmetric_todo_ans(x) == is_symmetric_str(x) for x in range(1, 10001))

# Bài 4
def count_prefix2(queries):
    top = max(h for _, h in queries)
    pre = [0] * (top + 1)
    for i in range(1, top + 1):
        pre[i] = pre[i - 1] + (1 if is_symmetric_todo_ans(i) else 0)
    return [pre[h] - pre[l - 1] for l, h in queries]

assert count_prefix2([(1, 100), (1200, 1230)]) == [9, 4]

# Bài 5(a)
def count_symmetric_general(low, high):
    count = 0
    for x in range(low, high + 1):
        s = str(x)
        if len(s) % 2 == 0:
            h = len(s) // 2
            if sum(map(int, s[:h])) == sum(map(int, s[h:])):
                count += 1
    return count

assert count_symmetric_general(1, 100) == 9
assert count_symmetric_general(1200, 1230) == 4
assert count_symmetric_general(100000, 100000) == 0      # 1+0+0 != 0+0+0
assert count_symmetric_general(100001, 100001) == 1      # 1+0+0 == 0+0+1
assert count_symmetric_general(10000, 10000) == 0        # 5 chữ số

# Bài 5(b)
def count_palindromes(low, high):
    return sum(1 for x in range(low, high + 1) if str(x) == str(x)[::-1])

assert count_palindromes(1, 100) == 18      # 1..9 và 11, 22, ..., 99
assert count_palindromes(100, 130) == 3     # 101, 111, 121

random.seed(13)
for _ in range(200):
    l = random.randint(1, 10000)
    h = random.randint(l, 10000)
    assert count_symmetric_general(l, h) == Solution().countSymmetricIntegers(l, h)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chỉ xét số có số chữ số chẵn; bóc chữ số bằng toán)

```python
class Solution:
    def countSymmetricIntegers(self, low: int, high: int) -> int:
        count = 0
        for x in range(low, high + 1):                       # đoạn đóng [low, high]
            if 10 <= x <= 99:
                count += (x // 10 == x % 10)
            elif 1000 <= x <= 9999:
                a, b, c, d = x // 1000, x // 100 % 10, x // 10 % 10, x % 10
                count += (a + b == c + d)
        return count
```

**Checklist**

- [ ] Số có **số chữ số lẻ** (1, 3, 5) không bao giờ đối xứng?
- [ ] So sánh **tổng** chữ số hai nửa, không so hai nửa giống nhau?
- [ ] Nửa sau có thể bắt đầu bằng `0` (như `1203`)?
- [ ] `range(low, high + 1)` (đoạn đóng)?
- [ ] `10000` có 5 chữ số, không đối xứng?
- [ ] Đã thử: `low = high`, `[1, 100]`, `[1, 10000]`, các biên `99`, `100`, `1000`?
- [ ] Đã so các cách với nhau trên đoạn ngẫu nhiên có seed?
- [ ] Signature đúng `countSymmetricIntegers(self, low: int, high: int) -> int`?